# GameTheory 16k — Théorie des enchères : le prix mesuré de l'information

**Navigation** : [<< 16c-Extraction-de-Revenu-DSIC-IR](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [Index GameTheory](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL-Python.ipynb)

**La théorie des enchères** (Myerson, Vickrey) : un vendeur met aux enchères un bien unique,
$n$ acquéreurs ont chacun une **valeur privée** qu'ils seuls connaissent. On **mesure** le
prix que l'asymétrie d'information impose au mécanisme — le symétrique exact de 16j
(screening / loyer d'information), du côté du vendeur.

À la famille **16 (Mécanismes)**, à côté de 16j : on calcule, pour des valeurs
$Uniforme[0,1]$, (1) la **vrais-facité** de l'enchère d'ordre 2 (Vickrey), (2) le **revenu
espéré du vendeur**, (3) la **réserve optimale de Myerson** et le **saut de revenu** qu'elle
produit, (4) le **théorème d'équivalence de revenu** (ordre 1 ≈ ordre 2).

Toutes les valeurs sont **calculées** dans le carnet (numpy, CPU seul, seedé), pas énoncées.


## 1. Le modèle

$n$ acquéreurs, chacun avec une **valeur privée** $v_i$ i.i.d. tirée de $Uniforme[0,1]$.
Le bien part au plus haut, sous une règle de paiement. Deux notions portent tout le reste :

| | Formule | Rôle |
|---|---|---|
| Fonction de répartition | $F(v) = v$  | probabilité qu'une valeur soit $\le v$ |
| Densité | $f(v) = 1$  | densité uniforme sur $[0,1]$ |
| **Valeur virtuelle** | $\varphi(v) = v - \dfrac{1-F(v)}{f(v)} = 2v - 1$  | le poids *économique* d'un acquéreur de valeur $v$ |

La **valeur virtuelle** est la clé de Myerson : en dessous de $r^*$ (où $\varphi=0$), un
acquéreur a une valeur virtuelle **négative** — le vendre, c'est créer de la rente d'information
qui coûte plus qu'il ne rapporte.


In [1]:
import numpy as np

# quadrature robuste : numpy 2.x -> trapezoid, numpy 1.x -> trapz (sans avertissement)
_trap = getattr(np, "trapezoid", None) or np.trapz

n = 3  # nombre d'acquéreurs

# Valeurs privées i.i.d. Uniforme[0, 1]
def F(v):  return np.clip(np.asarray(v, dtype=float), 0.0, 1.0)
def f(v):  return np.where((np.asarray(v, float) >= 0) & (np.asarray(v, float) <= 1), 1.0, 0.0)

# Valeur virtuelle de Myerson : phi(v) = v - (1 - F(v)) / f(v)
def phi(v):
    v = np.asarray(v, dtype=float)
    return v - (1.0 - F(v)) / np.where(f(v) > 0, f(v), 1.0)

# Reserve optimale : phi(r*) = 0  ->  r* = 1/2  pour Uniforme[0,1]
r_star = 0.5

print(f"n = {n} acqéreurs, valeurs i.i.d. Uniforme[0, 1]")
print(f"valeur virtuelle  phi(v) = 2v - 1 :")
for v in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"   v = {v:.2f}  ->  phi = {phi(v):.3f}")
print(f"reserve optimale  r* = {r_star}  (phi = 0 : en dessous, valeur virtuelle negative)")

n = 3 acqéreurs, valeurs i.i.d. Uniforme[0, 1]
valeur virtuelle  phi(v) = 2v - 1 :
   v = 0.00  ->  phi = -1.000
   v = 0.25  ->  phi = -0.500
   v = 0.50  ->  phi = 0.000
   v = 0.75  ->  phi = 0.500
   v = 1.00  ->  phi = 1.000
reserve optimale  r* = 0.5  (phi = 0 : en dessous, valeur virtuelle negative)


### Lecture

$\varphi$ croît de $-1$ (à $v=0$) à $+1$ (à $v=1$) et s'annule à $v=1/2$. Un acquéreur
dont la valeur est **sous** $1/2$ a une valeur virtuelle négative : si on le laisse remporter,
il paie moins que la rente d'information qu'il extrait. La réserve optimale **coupe** juste à
$1/2$ — c'est toute la décision de conception.


## 2. Vrais-facité de l'enchère d'ordre 2 (Vickrey)

En **enchère d'ordre 2** (au second prix), le gagnant paie la **plus haute offre des autres**
(dite $x$), pas la sienne. Soit $v$ la valeur qu'un acquéreur a réellement pour le bien, et $b$
son **offre**. Il remporte le bien **si et seulement si** $b \ge x$ ; s'il gagne, il paie $x$.
Son surplus est donc :

$$U(v; b, x) = \begin{cases} v - x & \text{si } b \ge x \\ 0 & \text{si } b < x . \end{cases}$$

On **mesure** que révéler sa vraie valeur ($b = v$) maximise ce surplus, **quelle que soit** la
compétition $x$ : l'offre franche est une **stratégie dominante**.


In [2]:
def surplus(v, x, b):
    return (v - x) if b >= x else 0.0

print("L'offre de sa VRAIE valeur v est-elle optimale ?  (ordre 2 / Vickrey)")
print(f"{'v':>5} | {'x (competition)':>15} | {'surplus si b=v':>16} | {'max de surplus':>16} | {'b=v optimal ?':>14}")
print("-" * 74)
for v, x in [(0.9, 0.4), (0.9, 0.8), (0.4, 0.3), (0.4, 0.6), (0.6, 0.6)]:
    b_grid = np.linspace(0, 1, 2001)
    surp_truthful = surplus(v, x, v)
    max_surp = max(surplus(v, x, b) for b in b_grid)
    optimal = "OUI" if abs(surp_truthful - max_surp) < 1e-9 else "non"
    print(f"{v:>5.2f} | {x:>15.2f} | {surp_truthful:>16.3f} | {max_surp:>16.3f} | {optimal:>14}")
print()
print("La revelation franche est une STRATEGIE DOMINANTE : optimale quelle que soit la competition x.")

L'offre de sa VRAIE valeur v est-elle optimale ?  (ordre 2 / Vickrey)
    v | x (competition) |   surplus si b=v |   max de surplus |  b=v optimal ?
--------------------------------------------------------------------------
 0.90 |            0.40 |            0.500 |            0.500 |            OUI
 0.90 |            0.80 |            0.100 |            0.100 |            OUI
 0.40 |            0.30 |            0.100 |            0.100 |            OUI
 0.40 |            0.60 |            0.000 |            0.000 |            OUI
 0.60 |            0.60 |            0.000 |            0.000 |            OUI

La revelation franche est une STRATEGIE DOMINANTE : optimale quelle que soit la competition x.


### Lecture

Sur chaque ligne, offrir sa vraie valeur $v$ atteint le surplus maximal : si $v > x$ elle gagne
et capte $v-x$ ; si $v < x$ elle perd (surplus $0$), ce qui est de toute façon le meilleur
possible. L'acquéreur **n'a pas besoin de modéliser ses adversaires** pour bien jouer — c'est
la propriété qui rend l'ordre 2 si précieux, et que 16j ne pouvait pas offrir.


## 3. Revenu du vendeur — ordre 2 sans réserve

Quand tous offrent franchement, le vendeur d'un ordre 2 **sans réserve** reçoit la **2ᵉ plus
haute valeur** des $n$ acquéreurs. C'est la **2ᵉ statistique d'ordre** $X_{(n-1)}$. Pour des
valeurs $Uniforme[0,1]$, son espérance a une forme close :

$$\mathbb{E}[X_{(n-1)}] = \frac{n-1}{n+1} .$$

On la **mesure** — formule close d'un côté, Monte-Carlo seedé de l'autre.


In [3]:
def rev_2price_noreserve_analytic(n):
    return (n - 1) / (n + 1)

def rev_2price_noreserve_mc(n, T=200000, seed=42):
    rg = np.random.default_rng(seed)
    vals = rg.random((T, n))
    # 2e plus haute valeur = index -2 apres tri croissant
    return vals[np.arange(T), np.argsort(vals, axis=1)[:, -2]].mean()

print("Revenu du vendeur en ordre 2 SANS reserve = E[2e statistique d'ordre]")
print(f"{'n':>4} | {'analytique (n-1)/(n+1)':>24} | {'Monte-Carlo (seed 42)':>22}")
print("-" * 54)
for n in [2, 3, 5, 10]:
    print(f"{n:>4} | {rev_2price_noreserve_analytic(n):>24.4f} | {rev_2price_noreserve_mc(n):>22.4f}")

Revenu du vendeur en ordre 2 SANS reserve = E[2e statistique d'ordre]
   n |   analytique (n-1)/(n+1) |  Monte-Carlo (seed 42)
------------------------------------------------------
   2 |                   0.3333 |                 0.3330
   3 |                   0.5000 |                 0.4995
   5 |                   0.6667 |                 0.6668
  10 |                   0.8182 |                 0.8182


### Lecture

La formule close et le Monte-Carlo coïncident à $10^{-4}$ près : le revenu de référence du
vendeur est **mesuré**, pas supposé. Pour $n=3$ il vaut $0.5000$ — c'est le **benchmark** que
la conception de mécanismes va chercher à dépasser.


## 4. La réserve de Myerson — le saut de revenu

Le **théorème de Myerson** : l'enchère qui maximise le revenu espéré du vendeur est un
**ordre 2 avec réserve** $r^*$, où $r^*$ est la valeur de **valeur virtuelle nulle**
($r^*=1/2$ ici). Règle : on vend **si** le plus haut bid atteint $r^*$, et le gagnant paie
$\max(r^*, \text{2ᵉ bid})$.

On **mesure** son revenu espéré — par intégrale (quadrature du triangle des deux plus hautes
valeurs) **et** par Monte-Carlo — et on le compare au benchmark sans réserve.


In [4]:
def rev_optimal_analytic(n, r):
    # E[ max(r, x) * 1{y >= r} ], x = 2e valeur, y = 1re, 0 <= x <= y <= 1.
    # Densite conjointe f(x,y) = n(n-1) x^(n-2). Integre en y d'abord :
    #   x >= r : max=r? non, max(r,x)=x, et y>=x>=r -> facteur (1-x)
    #   x <  r : max(r,x)=r, et y doit >= r     -> facteur (1-r)
    xs = np.linspace(0, 1, 20001)
    term = np.where(xs >= r, xs * (1 - xs), r * (1 - r))
    return _trap(term * n * (n - 1) * xs**(n - 2), xs)

def rev_optimal_mc(n, r, T=400000, seed=42):
    rg = np.random.default_rng(seed)
    v = rg.random((T, n))
    top2 = np.sort(v, axis=1)[:, -2:]            # [2e, 1re]
    payment = np.where(top2[:, 1] >= r, np.maximum(r, top2[:, 0]), 0.0)
    return payment.mean()

n = 3
base = rev_2price_noreserve_analytic(n)
opt  = rev_optimal_analytic(n, r_star)
opt_mc = rev_optimal_mc(n, r_star)
lift = opt - base
print(f"n = {n} acqéreurs, Uniforme[0,1], reserve r* = {r_star} :")
print(f"  ordre 2 SANS reserve (benchmark)             : {base:.5f}")
print(f"  ordre 2 AVEC reserve (optimal de Myerson)    : analytique = {opt:.5f}   Monte-Carlo = {opt_mc:.5f}")
print(f"  SAUT DE REVENU de la reserve                 : {lift:.5f}   (+{100*lift/base:.2f} %)")
assert abs(opt - opt_mc) < 5e-3, "integrale et Monte-Carlo divergent"


n = 3 acqéreurs, Uniforme[0,1], reserve r* = 0.5 :
  ordre 2 SANS reserve (benchmark)             : 0.50000
  ordre 2 AVEC reserve (optimal de Myerson)    : analytique = 0.53125   Monte-Carlo = 0.53076
  SAUT DE REVENU de la reserve                 : 0.03125   (+6.25 %)


### Lecture — le résultat central

Pour $n=3$, la réserve $1/2$ fait monter le revenu espéré de $0.50000$ à $0.53125$
(**$+6.25\%$**), et l'intégrale et le Monte-Carlo s'accordent. C'est le **prix mesuré de
l'information** du côté vendeur : la réserve enlève aux acquéreurs la rente d'information que
leur valeur supérieure à $r^*$ leur ferait autrement capturer. Modeste pour $n=3$ — et c'est
là que le §6 révèle l'essentiel.


## 5. Équivalence de revenu — ordre 1 ≈ ordre 2

En **enchère d'ordre 1** (au premier prix, le gagnant paie **son** bid), l'équilibre
symétrique est $b(v) = \frac{n-1}{n}\,v$. Le théorème d'équivalence de revenu dit que,
**en espérance pour le vendeur**, l'ordre 1 et l'ordre 2 (sans réserve) rapportent la même
chose. On le **mesure** : on simule l'ordre 1 à l'équilibre et on compare.


In [5]:
def rev_1price_mc(n, T=300000, seed=42):
    rg = np.random.default_rng(seed)
    v = rg.random((T, n))
    bids = (n - 1) / n * v                        # equilibre symetrique
    return bids[np.arange(T), np.argmax(bids, axis=1)].mean()

n = 3
r1 = rev_1price_mc(n)
r2 = rev_2price_noreserve_analytic(n)
print(f"n = {n} :  ordre 1 a l'equilibre (MC) = {r1:.4f}   |   ordre 2 sans reserve (analytique) = {r2:.4f}")
print(f"ecart = {abs(r1 - r2):.4f}   ->   THEOREME D'EQUIVALENCE DE REVENU (a la precision du Monte-Carlo)")

n = 3 :  ordre 1 a l'equilibre (MC) = 0.4999   |   ordre 2 sans reserve (analytique) = 0.5000
ecart = 0.0001   ->   THEOREME D'EQUIVALENCE DE REVENU (a la precision du Monte-Carlo)


### Lecture

L'ordre 1 (où chacun **sous-enchérit** pour ne pas payer trop) et l'ordre 2 (où chacun offre
franchement) drainent le **même** revenu espéré du vendeur. La *forme* de l'enchère change le
comportement, pas le revenu — l'unique bémol étant la réserve, qui est le levier de conception
vraiment efficace (c'est elle qui créait le saut du §4).


## 6. Sensibilité — le saut de réserve en fonction de $n$

La réserve rapporte-t-elle toujours la même chose ? On **mesure** le saut de revenu qu'elle
produit quand $n$ croît.


In [6]:
print(f"{'n':>4} | {'ordre 2 sans reserve':>20} | {'optimal (reserve 0.5)':>22} | {'saut':>9} | {'saut %':>9}")
print("-" * 72)
for n in [2, 3, 5, 8, 12]:
    base = rev_2price_noreserve_analytic(n)
    opt  = rev_optimal_analytic(n, r_star)
    lift = opt - base
    print(f"{n:>4} | {base:>20.4f} | {opt:>22.4f} | {lift:>9.4f} | {100*lift/base:>8.2f} %")
print()
print("La reserve rapporte d'autant plus que la competition est MAIGRE (n petit) :")
print("beaucoup d'acheteurs -> l'ordre 2 sans reserve extrait deja quasi tout le surplus.")

   n | ordre 2 sans reserve |  optimal (reserve 0.5) |      saut |    saut %
------------------------------------------------------------------------
   2 |               0.3333 |                 0.4167 |    0.0833 |    25.00 %
   3 |               0.5000 |                 0.5312 |    0.0312 |     6.25 %
   5 |               0.6667 |                 0.6719 |    0.0052 |     0.78 %
   8 |               0.7778 |                 0.7782 |    0.0004 |     0.06 %
  12 |               0.8462 |                 0.8462 |    0.0000 |     0.00 %

La reserve rapporte d'autant plus que la competition est MAIGRE (n petit) :
beaucoup d'acheteurs -> l'ordre 2 sans reserve extrait deja quasi tout le surplus.


### Lecture

Le saut de la réserve **fond** quand la compétition s'épaissit : $+25\%$ pour $n=2$,
$+6.25\%$ pour $n=3$, et presque rien pour $n=12$. Avec beaucoup d'acheteurs, la 2ᵉ plus
haute enchère monte déjà très haut et capture la rente ; la réserve n'a presque plus de marge.
**Concevoir une réserve est un levier quand le marché est mince, pas quand il est saturé.**


## 7. Exercices

### Exercice 1 — La réserve change de distribution

Remplacez $Uniforme[0,1]$ par une loi où $F(v)=v^2$ sur $[0,1]$ (densité $f(v)=2v$).
Recalculez la **valeur virtuelle** $\varphi(v)$ et la **réserve optimale** $r^*$.

*Indice : $\varphi(v) = v - (1-F(v))/f(v)$ ; $r^*$ est là où $\varphi=0$.*


In [7]:
# TODO etudiant — reserve optimale pour F(v)=v^2
r_star_beta = None  # TODO etudiant
print("Exercice a completer : reserve optimale F(v)=v^2  r* =", r_star_beta)

Exercice a completer : reserve optimale F(v)=v^2  r* =

 None


### Exercice 2 — Le paiement d'un gagnant
Dans un ordre 2 avec réserve $r^*=1/2$, trois acquéreurs offrent $0.30$, $0.80$, $0.55$.
Calculer : le gagnant, le **paiement** qu'il règle, et le **revenu** du vendeur.

*Indice : gagnant = plus haut bid ; paiement = $\max(r^*, 2^\text{e} \text{ bid})$ si le
gagnant $\ge r^*$, sinon pas de vente.*


In [8]:
# TODO etudiant — paiement du gagnant et revenu du vendeur
gagnant = None        # TODO etudiant
paiement = None       # TODO etudiant : max(r_star, 2e bid)
revenu_vendeur = None # TODO etudiant
print("Exercice a completer : gagnant =", gagnant, ", paiement =", paiement, ", revenu =", revenu_vendeur)

Exercice a completer : gagnant = None , paiement = None , revenu = None


### Exercice 3 — Le seuil de compétition
En lisant le tableau du §6, donner la plus petite valeur de $n$ pour laquelle le **saut de
revenu** de la réserve passe **sous** $1\%$.

*Indice : ce $n$ marque le point où la réserve devient un levier négligeable.*


In [9]:
# TODO etudiant — n a partir duquel le saut de la reserve < 1 %
n_seuil = None  # TODO etudiant
print("Exercice a completer : n seuil (saut < 1 %) =", n_seuil)

Exercice a completer : n seuil (saut < 1 %) = None


## 8. Ce que ce carnet établit

- **L'ordre 2 est vrai par construction** : révéler sa valeur est une stratégie dominante,
  mesurée ligne par ligne — l'acquéreur n'a pas à deviner ses rivaux.
- **L'asymétrie a un prix du côté vendeur** : la réserve de Myerson ($r^*=1/2$ ici) enlève la
  rente d'information et fait monter le revenu espéré — **mesuré** ici à $+6.25\%$ pour $n=3$.
- **Ce levier s'éteint avec la compétition** : le saut de la réserve fond quand $n$ croît
  ($+25\%$ à $n=2$, quasi nul à $n=12$) — la conception de la réserve est un art de
  **marché mince**.
- **La forme ne change pas le revenu** : ordre 1 et ordre 2 (sans réserve) sont équivalents en
  revenu espéré ; seule la réserve introduit un vrai gain.

C'est le miroir vendeur de 16j : là où 16j mesurait le **loyer d'information** que le
principal paie à l'agent informé, 16k mesure la **réserve** que le vendeur impose pour ne pas
payer cette rente — même théorème (Myerson), deux faces du même mécanisme.
